In [14]:
from pipelines.readers.b3_indices_segmentos_setoriais import ReaderSQLB3IndicesSegmentosSetoriais

from yfinance import download
import plotly.express as px

In [15]:
reader_b3_segmentos_setoriais = ReaderSQLB3IndicesSegmentosSetoriais()
df_b3_segmentos_setoriais = reader_b3_segmentos_setoriais.read(indice="IBEP")

print(df_b3_segmentos_setoriais.shape)
df_b3_segmentos_setoriais.tail(3)

(71, 7)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
68,IBEP,2026-07-06,VIVT3,TELEF BRASIL,ON EJ,707125712,1.166
69,IBEP,2026-07-06,WEGE3,WEG,ON NM,1485954732,3.278
70,IBEP,2026-07-06,YDUQ3,YDUQS PART,ON NM,261365845,0.112


In [16]:
tickers = list(df_b3_segmentos_setoriais["Código"] + ".SA") + ["^BVSP"]
series_precos = download(tickers, period="10y", interval="1d", progress=False, auto_adjust=False)['Adj Close']

In [17]:
series_precos.tail(3)

Ticker,ABEV3.SA,ALOS3.SA,ASAI3.SA,AURE3.SA,AXIA3.SA,AZZA3.SA,B3SA3.SA,BBDC3.SA,BBDC4.SA,BEEF3.SA,...,UGPA3.SA,USIM5.SA,VALE3.SA,VAMO3.SA,VBBR3.SA,VIVA3.SA,VIVT3.SA,WEGE3.SA,YDUQ3.SA,^BVSP
Date,,,,,,,,,,,,,,,,,,,,,
2026-07-02,16.299999,28.100000,8.69,11.66,54.250000,17.340000,14.61,15.535237,17.815004,3.51,...,26.600000,8.56,78.239998,2.80,29.830000,22.750000,34.610001,46.259998,8.92,172788.000000
2026-07-03,16.290001,28.200001,8.79,12.01,53.970001,17.139999,14.76,15.564641,17.913105,3.54,...,27.530001,8.77,78.839996,2.87,30.379999,22.770000,34.750000,46.480000,9.07,174266.000000
2026-07-06,15.880000,27.620001,8.67,12.24,53.540001,17.450001,14.58,15.550000,17.920000,3.51,...,27.940001,8.71,77.790001,2.87,30.120001,22.530001,34.500000,46.259998,8.69,172447.578125


In [18]:
retornos = series_precos.pct_change()

benchmark = retornos["^BVSP"]
ativos = retornos.drop(columns="^BVSP")

beta = ativos.apply(lambda col: col.cov(benchmark) / benchmark.var())

/tmp/ipykernel_27712/2971748431.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  retornos = series_precos.pct_change()


In [19]:
plot_df = (
    beta.rename("Beta")
        .rename_axis("Ticker")
        .reset_index()
        .sort_values("Beta")
)

fig = px.bar(
    plot_df,
    x="Ticker",
    y="Beta",
    color_discrete_sequence=["royalblue"],
    labels={
        "Ticker": "Empresa",
        "Beta": "Beta"
    },
    title="Beta por Empresa (vs IBOVESPA)"
)

# Linha de referência em β = 1
fig.add_hline(
    y=1,
    line_dash="dash",
    line_color="red",
    annotation_text="β = 1"
)

fig.update_layout(
    height=500,
    width=1200,
    xaxis_tickangle=-90
)

fig.show()

In [20]:
beta[beta.index == "VALE3.SA"]

Ticker
VALE3.SA    0.919886
dtype: float64

In [21]:
window = 60

mercado = retornos["^BVSP"]
ativo = retornos[ticker:="KLBN11.SA"]

rolling_beta = (
    ativo.rolling(window).cov(mercado)
    / mercado.rolling(window).var()
)

fig = px.line(
    rolling_beta,
    title=f"Rolling Beta ({window} períodos) - {ticker}",
)

fig.add_hline(y=1, line_dash="dash", line_color="red")

fig.show()